In [ ]:
# 初步查询实体的 文本描述 和 Wikidata_id
import requests
import time
import datetime
import csv
import os
import json
import re
from tqdm import tqdm  # 进度条库，如果没有安装请先运行: pip install tqdm
from difflib import SequenceMatcher

class WikidataBatchQuery:
    def __init__(self, input_file, output_file, progress_file="progress.json", delay=1.5):
        """
        初始化Wikidata批量查询
        
        参数:
            input_file: 包含实体列表的文本文件，每行一个实体
            output_file: 输出CSV文件路径
            progress_file: 进度保存文件路径
            delay: 请求间延迟（秒）
        """
        self.input_file = input_file
        self.output_file = output_file
        self.progress_file = progress_file
        self.delay = delay
        
        # 设置请求头
        self.headers = {
            'User-Agent': 'WikidataBatchQuery/1.0 (https://example.com; contact@example.com)'
        }
        
        # 时间模式正则表达式
        self.time_patterns = [
            r'\(\s*\d{4}\s*[-–—]\s*\d{4}\s*\)',  # (1929-2003)
            r'\(\s*\d{4}\s*\)',  # (1929)
            r'from\s+\d{4}\s+to\s+\d{4}',  # from 1929 to 2003
            r'since\s+\d{4}\s+to\s+\d{4}',  # since 1929 to 2003
            r'\d{4}\s*[-–—]\s*\d{4}',  # 1929-2003 (无括号)
            r'born\s+\d{4}',  # born 1929
            r'\b\d{4}\s*-\s*\d{4}\b',  # 1929 - 2003
            r'\b\d{4}\s*–\s*\d{4}\b',  # 1929 – 2003 (不同种类的横线)
            r'\b\d{4}\s*—\s*\d{4}\b',  # 1929 — 2003
        ]
        
        # 加载进度
        self.processed_entities = self.load_progress()
    
    def clean_description(self, description):
        """
        清理描述文本，移除时间相关的内容
        """
        if not description:
            return ""
        
        cleaned = description
        
        # 应用所有时间模式
        for pattern in self.time_patterns:
            cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE)
        
        # 清理多余的空格和标点
        cleaned = re.sub(r'\s+', ' ', cleaned)  # 多个空格变一个
        cleaned = re.sub(r'\s*,\s*,', ',', cleaned)  # 清理多余的逗号
        cleaned = re.sub(r'^\s*[,-]\s*', '', cleaned)  # 开头是逗号或横线
        cleaned = re.sub(r'\s*[,-]\s*$', '', cleaned)  # 结尾是逗号或横线
        cleaned = cleaned.strip()
        
        return cleaned.replace('()','')
    
    def load_progress(self):
        """加载已处理的实体和进度"""
        processed = set()
        
        # 从输出文件加载已处理的实体
        if os.path.exists(self.output_file):
            with open(self.output_file, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    processed.add(row['original_text'])
        
        # 从进度文件加载（用于断点续传）
        if os.path.exists(self.progress_file):
            try:
                with open(self.progress_file, 'r', encoding='utf-8') as f:
                    progress_data = json.load(f)
                    processed.update(progress_data.get('processed_entities', []))
            except:
                pass
                
        return processed
    
    def save_progress(self, current_entity=None):
        """保存进度到文件"""
        progress_data = {
            'processed_entities': list(self.processed_entities),
            'last_entity': current_entity,
            'timestamp': datetime.datetime.now().isoformat()
        }
        
        with open(self.progress_file, 'w', encoding='utf-8') as f:
            json.dump(progress_data, f, ensure_ascii=False, indent=2)
    
    def robust_wikidata_query(self, entity_text, max_retries=3):
        """
        优化的Wikidata查询函数，包含错误处理和重试机制
        返回：Wikidata ID, 标签和清理后的简短描述
        """
        wikidata_url = "https://www.wikidata.org/w/api.php"
        
        params = {
            'action': 'wbsearchentities',
            'search': entity_text,
            'language': 'en',
            'format': 'json',
            'limit': 5  # 减少结果数量以提高性能
        }
        
        for attempt in range(max_retries):
            try:
                response = requests.get(wikidata_url, params=params, headers=self.headers, timeout=30)
                
                if response.status_code == 200:
                    results = response.json().get('search', [])
                    if results:
                        # 获取最佳匹配的结果
                        best_result = results[0]
                        '''
                        best_result = None
                        best_score = 0
                        
                        for result in results:
                            score = SequenceMatcher(None, entity_text.lower(), result['label'].lower()).ratio()
                            if score > best_score:
                                if len(result.get('description', '')) != 0:
                                    best_score = score
                                    best_result = result
                        '''
                        qid = best_result['id']
                        label = best_result['label']

                        # 获取描述，如果没有描述则设为空字符串
                        description = best_result.get('description', '')
                        # 清理描述中的时间信息
                        cleaned_description = self.clean_description(description)
                        return qid, label, cleaned_description
                    return None, None, None
                    
                elif response.status_code == 429:
                    # 处理速率限制
                    retry_after = response.headers.get('retry-after', 10)
                    timeout = self.get_delay(retry_after)
                    print(f'\n速率限制，等待 {timeout} 秒后重试')
                    time.sleep(timeout)
                    continue
                    
                elif response.status_code == 403:
                    # 403错误可能需要更长时间等待
                    print(f"\n访问被拒绝，等待后重试")
                    if attempt < max_retries - 1:
                        wait_time = (2 ** attempt) * 30  # 更长的指数退避
                        print(f"等待 {wait_time} 秒后重试")
                        time.sleep(wait_time)
                    continue
                    
                else:
                    print(f"\nHTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(10)
                        
            except requests.exceptions.RequestException as e:
                print(f"\n请求异常: {e}")
                if attempt < max_retries - 1:
                    time.sleep(10)
        
        return None, None, None
    
    def get_delay(self, date_header):
        """
        从Retry-After头信息计算需要等待的时间
        """
        try:
            # 尝试解析HTTP日期格式
            date = datetime.datetime.strptime(date_header, '%a, %d %b %Y %H:%M:%S GMT')
            timeout = max(1, int((date - datetime.datetime.now()).total_seconds()))
        except ValueError:
            # 如果是秒数
            try:
                timeout = max(1, int(date_header))
            except ValueError:
                timeout = 10  # 默认等待10秒
        
        return timeout
    
    def process_entities(self):
        """处理所有实体"""
        # 读取实体列表
        e_2_idx = {}
        entities = []
        f = open(self.input_file, 'r', encoding='utf-8')
        for line in f.readlines():
            e_text, entity_idx_original, _, _ = line.strip().split('\t')
            clean_e_text = e_text.replace(' Inc.', '').replace(' Co.', '').replace(' Corp.', '').replace('Mrs. ', '').replace('Mr. ', '').replace('Ms. ', '').replace('Mx. ', '')
            e_2_idx[clean_e_text] = [int(entity_idx_original), e_text]
            entities.append(clean_e_text)
        print(e_2_idx)
        
        print(f"总实体数: {len(entities)}")
        print(f"已处理实体数: {len(self.processed_entities)}")
        print(f"待处理实体数: {len(entities) - len(self.processed_entities)}")
        
        # 过滤已处理的实体
        entities_to_process = [e for e in entities if e not in self.processed_entities]
        
        if not entities_to_process:
            print("所有实体已处理完成!")
            return
        
        # 创建或打开输出文件
        file_exists = os.path.exists(self.output_file)
        with open(self.output_file, 'a', newline='', encoding='utf-8') as f:
            fieldnames = ['original_idx_text', 'original_text', 'wikidata_id', 'wikidata_label', 'short_description']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            
            if not file_exists:
                writer.writeheader()
            
            # 使用进度条
            for i, entity in enumerate(tqdm(entities_to_process, desc="处理实体")):
                # 每30个请求后额外休息
                if i > 0 and i % 30 == 0:
                    print(f"\n已完成 {i} 个查询，休息60秒...")
                    time.sleep(60)
                
                # 查询实体
                qid, label, description = self.robust_wikidata_query(entity)
                
                # 写入结果
                writer.writerow({
                    'original_idx_text': e_2_idx[entity],
                    'original_text': entity,
                    'wikidata_id': qid if qid else '',
                    'wikidata_label': label if label else '',
                    'short_description': description if description else ''
                })
                f.flush()  # 确保数据写入磁盘
                
                # 更新进度
                self.processed_entities.add(entity)
                
                # 每10个实体保存一次进度
                if i % 10 == 0:
                    self.save_progress(entity)
                
                # 请求间延迟
                time.sleep(self.delay)
        
        # 处理完成后删除进度文件
        if os.path.exists(self.progress_file):
            os.remove(self.progress_file)
        
        print(f"\n批量查询完成! 结果已保存到: {self.output_file}")

def main():
    # 配置参数
    input_file = "entity2id.txt"  # 输入文件，每行一个实体
    output_file = "wikidata_results.csv"  # 输出CSV文件
    progress_file = "query_progress.json"  # 进度文件
    
    # 创建查询器并执行
    query = WikidataBatchQuery(
        input_file=input_file,
        output_file=output_file,
        progress_file=progress_file,
        delay=1.5  # 请求间延迟，可根据需要调整
    )
    
    # 开始处理
    query.process_entities()

# 测试清洗功能的函数
def test_cleaning():
    """测试描述清洗功能"""
    test_cases = [
        "American actor (1929-2003)",
        "American singer from 1929 to 2003",
        "British politician since 1929 to 2003",
        "French painter (1929)",
        "German composer 1929-2003",
        "Italian director born 1929",
        "Spanish writer 1929 – 2003",
        "Portuguese artist 1929—2003",
        "Normal description without dates",
        "Another normal description",
        ""
    ]
    
    query = WikidataBatchQuery("dummy.txt", "dummy.csv")
    
    print("测试描述清洗功能:")
    print("-" * 50)
    for test in test_cases:
        cleaned = query.clean_description(test)
        print(f"原描述: '{test}'")
        print(f"清洗后: '{cleaned}'")
        print("-" * 30)

if __name__ == "__main__":
    # 运行测试
    test_cleaning()
    
    # 运行主程序
    main()

In [ ]:
# 筛选空实体和失败实体
import csv
csv_reader = csv.reader(open("wikidata_results.csv"))
read_e = set()
empty_e = set()
for row in csv_reader:
	if row[0] == 'original_idx_text':
		continue
	read_e.add(eval(row[0])[1])
	if len(row[2]) ==0:
		empty_e.add(eval(row[0])[1])
print(len(empty_e))
print(len(read_e))
print(empty_e)

all_e = set()
input_file = "entity2id.txt"
f = open(input_file, 'r', encoding='utf-8')
for line in f.readlines():
	e_text, entity_idx_original, _, _ = line.strip().split('\t')
	all_e.add(e_text)
remain_e = all_e-read_e
print(len(remain_e))

In [ ]:
#对失败实体进一步细化查询 文本描述 和 Wikidata_id
import requests
import time
import datetime
import csv
import os
import json
import re
from tqdm import tqdm  # 进度条库，如果没有安装请先运行: pip install tqdm
from difflib import SequenceMatcher

class WikidataBatchQuery:
    def __init__(self, input_file, output_file, progress_file="progress.json", delay=1.5):
        """
        初始化Wikidata批量查询
        
        参数:
            input_file: 包含实体列表的文本文件，每行一个实体
            output_file: 输出CSV文件路径
            progress_file: 进度保存文件路径
            delay: 请求间延迟（秒）
        """
        self.input_file = input_file
        self.output_file = output_file
        self.progress_file = progress_file
        self.delay = delay
        
        # 设置请求头
        self.headers = {
            'User-Agent': 'WikidataBatchQuery/1.0 (https://example.com; contact@example.com)'
        }
        
        # 时间模式正则表达式
        self.time_patterns = [
            r'\(\s*\d{4}\s*[-–—]\s*\d{4}\s*\)',  # (1929-2003)
            r'\(\s*\d{4}\s*\)',  # (1929)
            r'from\s+\d{4}\s+to\s+\d{4}',  # from 1929 to 2003
            r'since\s+\d{4}\s+to\s+\d{4}',  # since 1929 to 2003
            r'\d{4}\s*[-–—]\s*\d{4}',  # 1929-2003 (无括号)
            r'born\s+\d{4}',  # born 1929
            r'\b\d{4}\s*-\s*\d{4}\b',  # 1929 - 2003
            r'\b\d{4}\s*–\s*\d{4}\b',  # 1929 – 2003 (不同种类的横线)
            r'\b\d{4}\s*—\s*\d{4}\b',  # 1929 — 2003
        ]
        
        # 加载进度
        self.processed_entities = self.load_progress()
    
    def clean_description(self, description):
        """
        清理描述文本，移除时间相关的内容
        """
        if not description:
            return ""
        
        cleaned = description
        
        # 应用所有时间模式
        for pattern in self.time_patterns:
            cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE)
        
        # 清理多余的空格和标点
        cleaned = re.sub(r'\s+', ' ', cleaned)  # 多个空格变一个
        cleaned = re.sub(r'\s*,\s*,', ',', cleaned)  # 清理多余的逗号
        cleaned = re.sub(r'^\s*[,-]\s*', '', cleaned)  # 开头是逗号或横线
        cleaned = re.sub(r'\s*[,-]\s*$', '', cleaned)  # 结尾是逗号或横线
        cleaned = cleaned.strip()
        
        return cleaned.replace('()','')
    
    def load_progress(self):
        """加载已处理的实体和进度"""
        processed = set()
        
        # 从输出文件加载已处理的实体
        if os.path.exists(self.output_file):
            with open(self.output_file, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    processed.add(row['original_text'])
        
        # 从进度文件加载（用于断点续传）
        if os.path.exists(self.progress_file):
            try:
                with open(self.progress_file, 'r', encoding='utf-8') as f:
                    progress_data = json.load(f)
                    processed.update(progress_data.get('processed_entities', []))
            except:
                pass
                
        return processed
    
    def save_progress(self, current_entity=None):
        """保存进度到文件"""
        progress_data = {
            'processed_entities': list(self.processed_entities),
            'last_entity': current_entity,
            'timestamp': datetime.datetime.now().isoformat()
        }
        
        with open(self.progress_file, 'w', encoding='utf-8') as f:
            json.dump(progress_data, f, ensure_ascii=False, indent=2)
    
    def robust_wikidata_query(self, entity_text, max_retries=3):
        """
        优化的Wikidata查询函数，包含错误处理和重试机制
        返回：Wikidata ID, 标签和清理后的简短描述
        """
        wikidata_url = "https://www.wikidata.org/w/api.php"
        
        params = {
            'action': 'wbsearchentities',
            'search': entity_text,
            'language': 'en',
            'format': 'json',
            'limit': 5  # 减少结果数量以提高性能
        }
        
        for attempt in range(max_retries):
            try:
                response = requests.get(wikidata_url, params=params, headers=self.headers, timeout=30)
                
                if response.status_code == 200:
                    results = response.json().get('search', [])
                    if results:
                        # 获取最佳匹配的结果
                        best_result = results[0]
                        '''
                        best_result = None
                        best_score = 0
                        
                        for result in results:
                            score = SequenceMatcher(None, entity_text.lower(), result['label'].lower()).ratio()
                            if score > best_score:
                                if len(result.get('description', '')) != 0:
                                    best_score = score
                                    best_result = result
                        '''
                        qid = best_result['id']
                        label = best_result['label']

                        # 获取描述，如果没有描述则设为空字符串
                        description = best_result.get('description', '')
                        # 清理描述中的时间信息
                        cleaned_description = self.clean_description(description)
                        return qid, label, cleaned_description
                    return None, None, None
                    
                elif response.status_code == 429:
                    # 处理速率限制
                    retry_after = response.headers.get('retry-after', 10)
                    timeout = self.get_delay(retry_after)
                    print(f'\n速率限制，等待 {timeout} 秒后重试')
                    time.sleep(timeout)
                    continue
                    
                elif response.status_code == 403:
                    # 403错误可能需要更长时间等待
                    print(f"\n访问被拒绝，等待后重试")
                    if attempt < max_retries - 1:
                        wait_time = (2 ** attempt) * 30  # 更长的指数退避
                        print(f"等待 {wait_time} 秒后重试")
                        time.sleep(wait_time)
                    continue
                    
                else:
                    print(f"\nHTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(10)
                        
            except requests.exceptions.RequestException as e:
                print(f"\n请求异常: {e}")
                if attempt < max_retries - 1:
                    time.sleep(10)
        
        return None, None, None
    
    def get_delay(self, date_header):
        """
        从Retry-After头信息计算需要等待的时间
        """
        try:
            # 尝试解析HTTP日期格式
            date = datetime.datetime.strptime(date_header, '%a, %d %b %Y %H:%M:%S GMT')
            timeout = max(1, int((date - datetime.datetime.now()).total_seconds()))
        except ValueError:
            # 如果是秒数
            try:
                timeout = max(1, int(date_header))
            except ValueError:
                timeout = 10  # 默认等待10秒
        
        return timeout
    
    def process_entities(self):
        """处理所有实体"""
        # 读取实体列表
        e_2_idx = {}
        entities = []
        f = open(self.input_file, 'r', encoding='utf-8')
        for line in f.readlines():
            e_text, entity_idx_original, _, _ = line.strip().split('\t')
            if e_text in read_e and e_text not in empty_e:
                continue
            clean_e_text = e_text.replace('Dr. ', '').replace(' Cos.', ' Companies').replace('Gov. ','').replace('Sen. ','').replace(' Ltd.', '').replace(' Inc.', '').replace(' Co.', '').replace(' Corp.', '').replace('Mrs. ', '').replace('Mr. ', '').replace('Ms. ', '').replace('Mx. ', '')
            e_2_idx[clean_e_text] = [int(entity_idx_original), e_text]
            entities.append(clean_e_text)
        
        print(f"总实体数: {len(entities)}")
        
        # 过滤已处理的实体
        entities_to_process = [e for e in entities]
        
        # 创建或打开输出文件
        file_exists = os.path.exists(self.output_file)
        with open(self.output_file, 'a', newline='', encoding='utf-8') as f:
            fieldnames = ['original_idx_text', 'original_text', 'wikidata_id', 'wikidata_label', 'short_description']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            
            if not file_exists:
                writer.writeheader()
            
            # 使用进度条
            for i, entity in enumerate(tqdm(entities_to_process, desc="处理实体")):
                # 每30个请求后额外休息
                if i > 0 and i % 30 == 0:
                    print(f"\n已完成 {i} 个查询，休息60秒...")
                    time.sleep(60)
                
                # 查询实体
                qid, label, description = self.robust_wikidata_query(entity)
                
                # 写入结果
                writer.writerow({
                    'original_idx_text': e_2_idx[entity],
                    'original_text': entity,
                    'wikidata_id': qid if qid else '',
                    'wikidata_label': label if label else '',
                    'short_description': description if description else ''
                })
                f.flush()  # 确保数据写入磁盘
                
                # 更新进度
                self.processed_entities.add(entity)
                
                # 每10个实体保存一次进度
                if i % 10 == 0:
                    self.save_progress(entity)
                
                # 请求间延迟
                time.sleep(self.delay)
        
        # 处理完成后删除进度文件
        if os.path.exists(self.progress_file):
            os.remove(self.progress_file)
        
        print(f"\n批量查询完成! 结果已保存到: {self.output_file}")

def main():
    # 配置参数
    input_file = "entity2id.txt"  # 输入文件，每行一个实体
    output_file = "wikidata_results.csv"  # 输出CSV文件
    progress_file = "query_progress.json"  # 进度文件
    
    # 创建查询器并执行
    query = WikidataBatchQuery(
        input_file=input_file,
        output_file=output_file,
        progress_file=progress_file,
        delay=1.5  # 请求间延迟，可根据需要调整
    )
    
    # 开始处理
    query.process_entities()

# 测试清洗功能的函数
def test_cleaning():
    """测试描述清洗功能"""
    test_cases = [
        "American actor (1929-2003)",
        "American singer from 1929 to 2003",
        "British politician since 1929 to 2003",
        "French painter (1929)",
        "German composer 1929-2003",
        "Italian director born 1929",
        "Spanish writer 1929 – 2003",
        "Portuguese artist 1929—2003",
        "Normal description without dates",
        "Another normal description",
        ""
    ]
    
    query = WikidataBatchQuery("dummy.txt", "dummy.csv")
    
    print("测试描述清洗功能:")
    print("-" * 50)
    for test in test_cases:
        cleaned = query.clean_description(test)
        print(f"原描述: '{test}'")
        print(f"清洗后: '{cleaned}'")
        print("-" * 30)

if __name__ == "__main__":
    # 运行测试
    test_cleaning()
    
    # 运行主程序
    main()

In [ ]:
# 通过Wikidata_id查询图谱中已有实体的时间区间知识（并行，但返回的是Wikidata_id，需要进一步抽取文本描述和label）
import requests
import json
import csv
import time
import re
import concurrent.futures
from tqdm import tqdm
from collections import defaultdict

class WikidataTemporalPropertiesOptimized:
    def __init__(self, delay=0.1, max_workers=5):
        """
        初始化Wikidata时间属性查询（优化版）
        
        参数:
            delay: 请求间延迟（秒）
            max_workers: 最大并行工作线程数
        """
        self.delay = delay
        self.max_workers = max_workers
        self.headers = {
            'User-Agent': 'WikidataTemporalProperties/2.0 (https://example.com; contact@example.com)'
        }
        
        # 定义时间相关属性
        self.start_time_prop = "P580"  # start time
        self.end_time_prop = "P582"    # end time
        
        # 实体和属性标签缓存
        self.label_cache = {}
        
        # 批量处理相关
        self.batch_size = 50  # 每批处理的实体数量
        
    def get_entities_batch(self, entity_ids, max_retries=3):
        """
        批量获取实体数据
        """
        if not entity_ids:
            return {}
            
        # 将实体ID列表转换为逗号分隔的字符串
        ids_str = "|".join(entity_ids)
        
        url = "https://www.wikidata.org/w/api.php"
        params = {
            'action': 'wbgetentities',
            'ids': ids_str,
            'format': 'json',
            'languages': 'en'
        }
        
        for attempt in range(max_retries):
            try:
                response = requests.get(url, params=params, headers=self.headers, timeout=60)
                
                if response.status_code == 200:
                    data = response.json()
                    return data.get('entities', {})
                elif response.status_code == 429:
                    # 处理速率限制
                    wait_time = 10 * (attempt + 1)
                    print(f"速率限制，等待{wait_time}秒后重试")
                    time.sleep(wait_time)
                    continue
                elif response.status_code == 404:
                    print(f"部分实体不存在")
                    return {}
                else:
                    print(f"HTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
            except requests.exceptions.RequestException as e:
                print(f"请求异常: {e}")
                if attempt < max_retries - 1:
                    time.sleep(5)
        
        return {}
    
    def get_labels_batch(self, entity_ids, max_retries=10):
        """
        批量获取实体标签
        """
        if not entity_ids:
            return {}
            
        # 过滤掉非实体ID
        q_ids = [eid for eid in entity_ids if eid.startswith('Q') or eid.startswith('P')]
        if not q_ids:
            return {}
            
        # 将实体ID列表转换为逗号分隔的字符串
        ids_str = "|".join(q_ids)
        
        url = "https://www.wikidata.org/w/api.php"
        params = {
            'action': 'wbgetentities',
            'ids': ids_str,
            'format': 'json',
            'languages': 'en',
            'props': 'labels'
        }
        
        for attempt in range(max_retries):
                response = requests.get(url, params=params, headers=self.headers, timeout=60)
                print(response)
                if response.status_code == 200:
                    data = response.json()
                    entities = data.get('entities', {})
                    
                    labels = {}
                    for eid, entity in entities.items():
                        labels_data = entity.get('labels', {})
                        if 'en' in labels_data:
                            labels[eid] = labels_data['en'].get('value', eid)
                        else:
                            # 如果没有英文标签，使用第一个可用标签
                            for lang in labels_data:
                                if labels_data[lang].get('value'):
                                    labels[eid] = labels_data[lang].get('value')
                                    break
                            else:
                                labels[eid] = eid
                    
                    # 更新缓存
                    self.label_cache.update(labels)
                    return labels
                elif response.status_code == 429:
                    # 处理速率限制
                    wait_time = 10 * (attempt + 1)
                    print(f"速率限制，等待{wait_time}秒后重试")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"获取标签时HTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
                '''
                except requests.exceptions.RequestException as e:
                    print(f"获取标签时请求异常: {e}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
                '''
        
        return {}
    
    def extract_temporal_claims(self, entity_data):
        """
        从实体数据中提取包含时间信息的声明
        """
        temporal_claims = []
        
        if not entity_data or 'entities' not in entity_data:
            return temporal_claims
        
        # 收集所有需要获取标签的实体ID
        entities_to_label = set()
        
        for entity_id, entity in entity_data['entities'].items():
            # 添加头实体ID到标签查询列表
            entities_to_label.add(entity_id)
            
            if 'claims' not in entity:
                continue
                
            # 遍历所有属性
            for prop_id, claims in entity['claims'].items():
                entities_to_label.add(prop_id)  # 添加属性ID
                
                for claim in claims:
                    # 检查声明是否有限定符
                    if 'qualifiers' in claim:
                        qualifiers = claim['qualifiers']
                        
                        # 检查是否有start time或end time限定符
                        has_start_time = self.start_time_prop in qualifiers
                        has_end_time = self.end_time_prop in qualifiers
                        
                        if has_start_time or has_end_time:
                            # 获取主声明值
                            mainsnak = claim.get('mainsnak', {})
                            datatype = mainsnak.get('datatype', '')
                            datavalue = mainsnak.get('datavalue', {})
                            
                            # 跳过数字类型的值
                            if datatype == 'quantity':
                                continue
                            
                            # 提取主声明值
                            main_value_id, main_value_type = self.extract_main_value_info(mainsnak)
                            if main_value_id and main_value_id.startswith(('Q', 'P')):
                                entities_to_label.add(main_value_id)
                            
                            # 提取时间限定符
                            start_times = self.extract_time_values(qualifiers.get(self.start_time_prop, []))
                            end_times = self.extract_time_values(qualifiers.get(self.end_time_prop, []))
                            
                            temporal_claims.append({
                                'entity_id': entity_id,
                                'property_id': prop_id,
                                'main_value_id': main_value_id,
                                'main_value_type': main_value_type,
                                'start_times': start_times,
                                'end_times': end_times,
                                'has_start_time': has_start_time,
                                'has_end_time': has_end_time,
                                'time_status': self.get_time_status(has_start_time, has_end_time)
                            })
        
        # 批量获取所有需要的标签
        self.get_labels_batch(list(entities_to_label))
        
        # 为每个声明添加标签
        for claim in temporal_claims:
            # 头实体标签
            claim['entity_label'] = self.label_cache.get(claim['entity_id'], claim['entity_id'])
            
            # 属性标签
            claim['property_label'] = self.label_cache.get(claim['property_id'], claim['property_id'])
            
            # 尾实体标签
            main_value_id = claim['main_value_id']
            if main_value_id and main_value_id.startswith(('Q', 'P')):
                claim['main_value_label'] = self.label_cache.get(main_value_id, main_value_id)
            else:
                claim['main_value_label'] = main_value_id
        
        return temporal_claims
    
    def extract_main_value_info(self, mainsnak):
        """
        提取主声明的值和类型
        返回: (值ID, 值类型)
        """
        if not mainsnak.get('datavalue'):
            return None, None
        
        datavalue = mainsnak['datavalue']
        value_type = datavalue.get('type', '')
        value = datavalue.get('value', '')
        
        if value_type == 'wikibase-entityid':
            return value.get('id', ''), 'entity'
        elif value_type == 'string':
            return value, 'string'
        elif value_type == 'time':
            time_value = value.get('time', '')
            # 清理时间格式：去除+号，只保留日期部分
            if time_value:
                time_value = re.sub(r'^\+(.*)T.*', r'\1', time_value)
            return time_value, 'time'
        elif value_type == 'monolingualtext':
            text = value.get('text', '')
            return text, 'text'
        else:
            value_str = str(value)
            return value_str, 'other'
    
    def extract_time_values(self, time_snaks):
        """
        提取时间值
        """
        time_values = []
        for snak in time_snaks:
            if snak.get('datavalue'):
                datavalue = snak['datavalue']
                if datavalue.get('type') == 'time':
                    time_value = datavalue.get('value', {})
                    # 提取时间字符串（去除+符号和时区信息）
                    time_str = time_value.get('time', '')
                    if time_str:
                        # 清理时间格式：去除+号，只保留日期部分
                        time_str = re.sub(r'^\+(.*)T.*', r'\1', time_str)
                        time_values.append(time_str)
        return time_values
    
    def get_time_status(self, has_start, has_end):
        """
        获取时间状态描述
        """
        if has_start and has_end:
            return "both"
        elif has_start and not has_end:
            return "start_only"
        elif not has_start and has_end:
            return "end_only"
        else:
            return "none"
    
    def process_entities_parallel(self, entity_ids, output_file):
        """
        使用并行处理处理实体列表
        """
        # 将实体ID分成批次
        batches = [entity_ids[i:i+self.batch_size] for i in range(0, len(entity_ids), self.batch_size)]
        
        print(f"将处理 {len(entity_ids)} 个实体，分为 {len(batches)} 批，每批 {self.batch_size} 个")
        print(f"使用 {self.max_workers} 个并行工作线程")
        
        all_temporal_claims = []
        
        # 使用线程池并行处理批次
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            # 提交所有批次任务
            future_to_batch = {executor.submit(self.process_single_batch, batch): batch for batch in batches}
            
            # 收集结果
            for future in tqdm(concurrent.futures.as_completed(future_to_batch), 
                              total=len(batches), desc="处理批次"):
                batch_claims = future.result()
                all_temporal_claims.extend(batch_claims)
        
        # 保存结果到CSV文件
        self.save_results(all_temporal_claims, output_file)
        
        return all_temporal_claims
    
    def process_single_batch(self, batch):
        """
        处理单个批次
        """
        # 批量获取实体数据
        entity_data = self.get_entities_batch(batch)
        
        if not entity_data:
            return []
        
        # 提取时间声明
        temporal_claims = self.extract_temporal_claims({'entities': entity_data})
        
        return temporal_claims
    
    def save_results(self, temporal_claims, output_file):
        """
        保存结果到CSV文件
        """
        if not temporal_claims:
            print("没有找到时间属性声明")
            return
        
        with open(output_file, 'w+', newline='', encoding='utf-8') as f:
            fieldnames = [
                'head_entity_id',      # 头实体ID
                'head_entity_label',   # 头实体标签
                'property_id',         # 属性ID
                'property_label',      # 属性标签
                'tail_entity_id',      # 尾实体ID
                'tail_entity_label',   # 尾实体标签
                'value_type',          # 值类型
                'start_times',         # 开始时间
                'end_times',           # 结束时间
                'time_status'          # 时间状态
            ]
            
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            
            for claim in temporal_claims:
                writer.writerow({
                    'head_entity_id': claim['entity_id'],
                    'head_entity_label': claim['entity_label'],
                    'property_id': claim['property_id'],
                    'property_label': claim['property_label'],
                    'tail_entity_id': claim['main_value_id'],
                    'tail_entity_label': claim['main_value_label'],
                    'value_type': claim['main_value_type'],
                    'start_times': '; '.join(claim['start_times']),
                    'end_times': '; '.join(claim['end_times']),
                    'time_status': claim['time_status']
                })
        
        print(f"结果已保存到: {output_file}")
        print(f"共找到 {len(temporal_claims)} 个时间属性声明")

def main():
    # 示例使用
    entity_ids = [
        "Q76",    # Barack Obama
        "Q90",    # Paris
        "Q114",   # Kenya
        "Q937",   # Albert Einstein
        "Q1339",  # Leonardo da Vinci
        "Q1774"   # Mozart
    ]
    
    output_file = "wikidata_temporal_properties.csv"
    
    # 创建查询器
    query = WikidataTemporalPropertiesOptimized(delay=0.1, max_workers=5)
    
    # 处理实体
    results = query.process_entities_parallel(entity_ids, output_file)
    
    # 打印摘要
    print("\n处理摘要:")
    print(f"处理的实体数量: {len(entity_ids)}")
    print(f"找到的时间属性声明数量: {len(results)}")
    
    # 按时间状态分组
    status_counts = {}
    for claim in results:
        status = claim['time_status']
        status_counts[status] = status_counts.get(status, 0) + 1
    
    print("\n时间状态分布:")
    for status, count in status_counts.items():
        print(f"  {status}: {count}")

def load_entity_ids_from_file(file_path):
    """
    从文件加载实体ID列表
    """
    csv_reader = csv.reader(open("wikidata_results.csv"))
    read_Q_2_original_e = {}

    for row in csv_reader:
        if row[0] == 'original_idx_text':
            continue
        Q_id = row[2]
        original_e = eval(row[0])[1]
        if Q_id not in read_Q_2_original_e.keys():
            read_Q_2_original_e[Q_id] = original_e
    print(len(read_Q_2_original_e.keys()))
    
    # 确保是有效的Wikidata ID格式
    entity_ids = []
    for entity in read_Q_2_original_e.keys():
        if entity.startswith('Q') and entity[1:].isdigit():
            entity_ids.append(entity)
        else:
            print(f"跳过无效的实体ID: {entity}")
    print(len(entity_ids))
    return entity_ids

if __name__ == "__main__":
    #main()
    
    
    # 从文件加载实体ID
    input_file = "wikidata_results.csv"  # 包含QID的文件，每行一个
    output_file = "wikidata_temporal_properties.csv"
    
    # 加载实体ID
    entity_ids = load_entity_ids_from_file(input_file)
    print(entity_ids[:10])
    
    if not entity_ids:
        print("没有找到有效的实体ID")
        exit()
    
    print(f"加载了 {len(entity_ids)} 个实体ID")
    
    # 创建查询器并处理
    query = WikidataTemporalPropertiesOptimized(delay=0.1, max_workers=10)
    results = query.process_entities_parallel(entity_ids, output_file)

    # 打印摘要
    print("\n处理摘要:")
    print(f"处理的实体数量: {len(entity_ids)}")
    print(f"找到的时间属性声明数量: {len(results)}")
    
    # 按时间状态分组
    status_counts = {}
    for claim in results:
        status = claim['time_status']
        status_counts[status] = status_counts.get(status, 0) + 1
    
    print("\n时间状态分布:")
    for status, count in status_counts.items():
        print(f"  {status}: {count}")
    
    

In [ ]:
# 获取可用的时间区间知识
csv_reader = csv.reader(open("wikidata_temporal_properties.csv"))
span_knowledge = set()
new_e_set = set()
new_r_set = set()
facts_with_nature_end = set()

for row in csv_reader:
    if row[0] == 'head_entity_id':
        continue
    sub_id = row[0]
    rel_id = row[2]
    obj_id = row[4]

    obj_type = row[6]
    start_time = row[7]
    end_time = row[8]

    if obj_type != 'entity' or (start_time == '' and end_time == ''):
        continue
    if start_time != '' and end_time == '':
        if start_time <= '2023-01-01':
            if start_time >= '2018-01-07':
                span_knowledge.add((sub_id, rel_id, obj_id, start_time, '2023-01-01'))
                new_e_set.add(obj_id)
                new_r_set.add(rel_id)
            else:
                span_knowledge.add((sub_id, rel_id, obj_id, '2018-01-07', '2023-01-01'))
                new_e_set.add(obj_id)
                new_r_set.add(rel_id)
    
    if start_time == '' and end_time != '':
        if end_time >= '2018-01-07':
            if end_time <= '2023-01-01':
                span_knowledge.add((sub_id, rel_id, obj_id, '2018-01-07', end_time))
                new_e_set.add(obj_id)
                new_r_set.add(rel_id)
                facts_with_nature_end.add((sub_id, rel_id, obj_id, '2018-01-07', end_time))
            else:
                span_knowledge.add((sub_id, rel_id, obj_id, '2018-01-07', '2023-01-01'))
                new_e_set.add(obj_id)
                new_r_set.add(rel_id)
    
    if start_time != '' and end_time != '':
        new_start_time, new_end_time = -1, -1
        if start_time <= '2023-01-01' and start_time >= '2018-01-07':
            new_start_time = start_time
        elif start_time < '2018-01-07':
            new_start_time = '2018-01-07'
        
        if end_time <= '2023-01-01' and end_time >= '2018-01-07':
            new_end_time = end_time
        elif end_time > '2023-01-01':
            new_end_time = '2023-01-01'
        
        if new_start_time != -1 and new_end_time != -1:
            span_knowledge.add((sub_id, rel_id, obj_id, new_start_time, new_end_time))
            new_e_set.add(obj_id)
            new_r_set.add(rel_id)
            if new_end_time != '2023-01-01':
                facts_with_nature_end.add((sub_id, rel_id, obj_id, new_start_time, new_end_time))

print(len(span_knowledge))
print(len(facts_with_nature_end))
print(len(new_e_set))
print(len(new_r_set))
print(new_e_set)
print(new_r_set)
print(list(span_knowledge)[:10])

In [ ]:
# 进一步为上一步时间区间知识尾实体抽取时间区间知识
# 通过Wikidata_id查询图谱中已有实体的时间区间知识（并行，但返回的是Wikidata_id，需要进一步抽取文本描述和label）
import requests
import json
import csv
import time
import re
import concurrent.futures
from tqdm import tqdm
from collections import defaultdict

class WikidataTemporalPropertiesOptimized:
    def __init__(self, delay=0.1, max_workers=5):
        """
        初始化Wikidata时间属性查询（优化版）
        
        参数:
            delay: 请求间延迟（秒）
            max_workers: 最大并行工作线程数
        """
        self.delay = delay
        self.max_workers = max_workers
        self.headers = {
            'User-Agent': 'WikidataTemporalProperties/2.0 (https://example.com; contact@example.com)'
        }
        
        # 定义时间相关属性
        self.start_time_prop = "P580"  # start time
        self.end_time_prop = "P582"    # end time
        
        # 实体和属性标签缓存
        self.label_cache = {}
        
        # 批量处理相关
        self.batch_size = 50  # 每批处理的实体数量
        
    def get_entities_batch(self, entity_ids, max_retries=3):
        """
        批量获取实体数据
        """
        if not entity_ids:
            return {}
            
        # 将实体ID列表转换为逗号分隔的字符串
        ids_str = "|".join(entity_ids)
        
        url = "https://www.wikidata.org/w/api.php"
        params = {
            'action': 'wbgetentities',
            'ids': ids_str,
            'format': 'json',
            'languages': 'en'
        }
        
        for attempt in range(max_retries):
            try:
                response = requests.get(url, params=params, headers=self.headers, timeout=60)
                
                if response.status_code == 200:
                    data = response.json()
                    return data.get('entities', {})
                elif response.status_code == 429:
                    # 处理速率限制
                    wait_time = 10 * (attempt + 1)
                    print(f"速率限制，等待{wait_time}秒后重试")
                    time.sleep(wait_time)
                    continue
                elif response.status_code == 404:
                    print(f"部分实体不存在")
                    return {}
                else:
                    print(f"HTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
            except requests.exceptions.RequestException as e:
                print(f"请求异常: {e}")
                if attempt < max_retries - 1:
                    time.sleep(5)
        
        return {}
    
    def get_labels_batch(self, entity_ids, max_retries=10):
        """
        批量获取实体标签
        """
        if not entity_ids:
            return {}
            
        # 过滤掉非实体ID
        q_ids = [eid for eid in entity_ids if eid.startswith('Q') or eid.startswith('P')]
        if not q_ids:
            return {}
            
        # 将实体ID列表转换为逗号分隔的字符串
        ids_str = "|".join(q_ids)
        
        url = "https://www.wikidata.org/w/api.php"
        params = {
            'action': 'wbgetentities',
            'ids': ids_str,
            'format': 'json',
            'languages': 'en',
            'props': 'labels'
        }
        
        for attempt in range(max_retries):
                response = requests.get(url, params=params, headers=self.headers, timeout=60)
                print(response)
                if response.status_code == 200:
                    data = response.json()
                    entities = data.get('entities', {})
                    
                    labels = {}
                    for eid, entity in entities.items():
                        labels_data = entity.get('labels', {})
                        if 'en' in labels_data:
                            labels[eid] = labels_data['en'].get('value', eid)
                        else:
                            # 如果没有英文标签，使用第一个可用标签
                            for lang in labels_data:
                                if labels_data[lang].get('value'):
                                    labels[eid] = labels_data[lang].get('value')
                                    break
                            else:
                                labels[eid] = eid
                    
                    # 更新缓存
                    self.label_cache.update(labels)
                    return labels
                elif response.status_code == 429:
                    # 处理速率限制
                    wait_time = 10 * (attempt + 1)
                    print(f"速率限制，等待{wait_time}秒后重试")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"获取标签时HTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
                '''
                except requests.exceptions.RequestException as e:
                    print(f"获取标签时请求异常: {e}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
                '''
        
        return {}
    
    def extract_temporal_claims(self, entity_data):
        """
        从实体数据中提取包含时间信息的声明
        """
        temporal_claims = []
        
        if not entity_data or 'entities' not in entity_data:
            return temporal_claims
        
        # 收集所有需要获取标签的实体ID
        entities_to_label = set()
        
        for entity_id, entity in entity_data['entities'].items():
            # 添加头实体ID到标签查询列表
            entities_to_label.add(entity_id)
            
            if 'claims' not in entity:
                continue
                
            # 遍历所有属性
            for prop_id, claims in entity['claims'].items():
                entities_to_label.add(prop_id)  # 添加属性ID
                
                for claim in claims:
                    # 检查声明是否有限定符
                    if 'qualifiers' in claim:
                        qualifiers = claim['qualifiers']
                        
                        # 检查是否有start time或end time限定符
                        has_start_time = self.start_time_prop in qualifiers
                        has_end_time = self.end_time_prop in qualifiers
                        
                        if has_start_time or has_end_time:
                            # 获取主声明值
                            mainsnak = claim.get('mainsnak', {})
                            datatype = mainsnak.get('datatype', '')
                            datavalue = mainsnak.get('datavalue', {})
                            
                            # 跳过数字类型的值
                            if datatype == 'quantity':
                                continue
                            
                            # 提取主声明值
                            main_value_id, main_value_type = self.extract_main_value_info(mainsnak)
                            if main_value_id and main_value_id.startswith(('Q', 'P')):
                                entities_to_label.add(main_value_id)
                            
                            # 提取时间限定符
                            start_times = self.extract_time_values(qualifiers.get(self.start_time_prop, []))
                            end_times = self.extract_time_values(qualifiers.get(self.end_time_prop, []))
                            
                            temporal_claims.append({
                                'entity_id': entity_id,
                                'property_id': prop_id,
                                'main_value_id': main_value_id,
                                'main_value_type': main_value_type,
                                'start_times': start_times,
                                'end_times': end_times,
                                'has_start_time': has_start_time,
                                'has_end_time': has_end_time,
                                'time_status': self.get_time_status(has_start_time, has_end_time)
                            })
        
        # 批量获取所有需要的标签
        self.get_labels_batch(list(entities_to_label))
        
        # 为每个声明添加标签
        for claim in temporal_claims:
            # 头实体标签
            claim['entity_label'] = self.label_cache.get(claim['entity_id'], claim['entity_id'])
            
            # 属性标签
            claim['property_label'] = self.label_cache.get(claim['property_id'], claim['property_id'])
            
            # 尾实体标签
            main_value_id = claim['main_value_id']
            if main_value_id and main_value_id.startswith(('Q', 'P')):
                claim['main_value_label'] = self.label_cache.get(main_value_id, main_value_id)
            else:
                claim['main_value_label'] = main_value_id
        
        return temporal_claims
    
    def extract_main_value_info(self, mainsnak):
        """
        提取主声明的值和类型
        返回: (值ID, 值类型)
        """
        if not mainsnak.get('datavalue'):
            return None, None
        
        datavalue = mainsnak['datavalue']
        value_type = datavalue.get('type', '')
        value = datavalue.get('value', '')
        
        if value_type == 'wikibase-entityid':
            return value.get('id', ''), 'entity'
        elif value_type == 'string':
            return value, 'string'
        elif value_type == 'time':
            time_value = value.get('time', '')
            # 清理时间格式：去除+号，只保留日期部分
            if time_value:
                time_value = re.sub(r'^\+(.*)T.*', r'\1', time_value)
            return time_value, 'time'
        elif value_type == 'monolingualtext':
            text = value.get('text', '')
            return text, 'text'
        else:
            value_str = str(value)
            return value_str, 'other'
    
    def extract_time_values(self, time_snaks):
        """
        提取时间值
        """
        time_values = []
        for snak in time_snaks:
            if snak.get('datavalue'):
                datavalue = snak['datavalue']
                if datavalue.get('type') == 'time':
                    time_value = datavalue.get('value', {})
                    # 提取时间字符串（去除+符号和时区信息）
                    time_str = time_value.get('time', '')
                    if time_str:
                        # 清理时间格式：去除+号，只保留日期部分
                        time_str = re.sub(r'^\+(.*)T.*', r'\1', time_str)
                        time_values.append(time_str)
        return time_values
    
    def get_time_status(self, has_start, has_end):
        """
        获取时间状态描述
        """
        if has_start and has_end:
            return "both"
        elif has_start and not has_end:
            return "start_only"
        elif not has_start and has_end:
            return "end_only"
        else:
            return "none"
    
    def process_entities_parallel(self, entity_ids, output_file):
        """
        使用并行处理处理实体列表
        """
        # 将实体ID分成批次
        batches = [entity_ids[i:i+self.batch_size] for i in range(0, len(entity_ids), self.batch_size)]
        
        print(f"将处理 {len(entity_ids)} 个实体，分为 {len(batches)} 批，每批 {self.batch_size} 个")
        print(f"使用 {self.max_workers} 个并行工作线程")
        
        all_temporal_claims = []
        
        # 使用线程池并行处理批次
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            # 提交所有批次任务
            future_to_batch = {executor.submit(self.process_single_batch, batch): batch for batch in batches}
            
            # 收集结果
            for future in tqdm(concurrent.futures.as_completed(future_to_batch), 
                              total=len(batches), desc="处理批次"):
                batch_claims = future.result()
                all_temporal_claims.extend(batch_claims)
        
        # 保存结果到CSV文件
        self.save_results(all_temporal_claims, output_file)
        
        return all_temporal_claims
    
    def process_single_batch(self, batch):
        """
        处理单个批次
        """
        # 批量获取实体数据
        entity_data = self.get_entities_batch(batch)
        
        if not entity_data:
            return []
        
        # 提取时间声明
        temporal_claims = self.extract_temporal_claims({'entities': entity_data})
        
        return temporal_claims
    
    def save_results(self, temporal_claims, output_file):
        """
        保存结果到CSV文件
        """
        if not temporal_claims:
            print("没有找到时间属性声明")
            return
        
        with open(output_file, 'a', newline='', encoding='utf-8') as f:
            fieldnames = [
                'head_entity_id',      # 头实体ID
                'head_entity_label',   # 头实体标签
                'property_id',         # 属性ID
                'property_label',      # 属性标签
                'tail_entity_id',      # 尾实体ID
                'tail_entity_label',   # 尾实体标签
                'value_type',          # 值类型
                'start_times',         # 开始时间
                'end_times',           # 结束时间
                'time_status'          # 时间状态
            ]
            
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            
            for claim in temporal_claims:
                writer.writerow({
                    'head_entity_id': claim['entity_id'],
                    'head_entity_label': claim['entity_label'],
                    'property_id': claim['property_id'],
                    'property_label': claim['property_label'],
                    'tail_entity_id': claim['main_value_id'],
                    'tail_entity_label': claim['main_value_label'],
                    'value_type': claim['main_value_type'],
                    'start_times': '; '.join(claim['start_times']),
                    'end_times': '; '.join(claim['end_times']),
                    'time_status': claim['time_status']
                })
        
        print(f"结果已保存到: {output_file}")
        print(f"共找到 {len(temporal_claims)} 个时间属性声明")

def main():
    # 示例使用
    entity_ids = [
        "Q76",    # Barack Obama
        "Q90",    # Paris
        "Q114",   # Kenya
        "Q937",   # Albert Einstein
        "Q1339",  # Leonardo da Vinci
        "Q1774"   # Mozart
    ]
    
    output_file = "wikidata_temporal_properties.csv"
    
    # 创建查询器
    query = WikidataTemporalPropertiesOptimized(delay=0.1, max_workers=5)
    
    # 处理实体
    results = query.process_entities_parallel(entity_ids, output_file)
    
    # 打印摘要
    print("\n处理摘要:")
    print(f"处理的实体数量: {len(entity_ids)}")
    print(f"找到的时间属性声明数量: {len(results)}")
    
    # 按时间状态分组
    status_counts = {}
    for claim in results:
        status = claim['time_status']
        status_counts[status] = status_counts.get(status, 0) + 1
    
    print("\n时间状态分布:")
    for status, count in status_counts.items():
        print(f"  {status}: {count}")

def load_entity_ids_from_file(file):
    """
    从文件加载实体ID列表
    """
    # 确保是有效的Wikidata ID格式
    entity_ids = []
    for entity in file:
        if entity.startswith('Q') and entity[1:].isdigit():
            entity_ids.append(entity)
        else:
            print(f"跳过无效的实体ID: {entity}")
    print(len(entity_ids))
    return entity_ids

if __name__ == "__main__":
    #main()
    
    # 从文件加载实体ID
    output_file = "wikidata_temporal_properties.csv"
    
    # 加载实体ID
    entity_ids = load_entity_ids_from_file(new_e_set)
    print(entity_ids[:10])
    
    if not entity_ids:
        print("没有找到有效的实体ID")
        exit()
    
    print(f"加载了 {len(entity_ids)} 个实体ID")
    
    # 创建查询器并处理
    query = WikidataTemporalPropertiesOptimized(delay=0.1, max_workers=10)
    results = query.process_entities_parallel(entity_ids, output_file)

    # 打印摘要
    print("\n处理摘要:")
    print(f"处理的实体数量: {len(entity_ids)}")
    print(f"找到的时间属性声明数量: {len(results)}")
    
    # 按时间状态分组
    status_counts = {}
    for claim in results:
        status = claim['time_status']
        status_counts[status] = status_counts.get(status, 0) + 1
    
    print("\n时间状态分布:")
    for status, count in status_counts.items():
        print(f"  {status}: {count}")
    
    

In [ ]:
# 获取可用的时间区间知识
csv_reader = csv.reader(open("wikidata_temporal_properties.csv"))
span_knowledge = set()
new_e_set = set()
new_r_set = set()
facts_with_nature_end = set()

for row in csv_reader:
    if row[0] == 'head_entity_id':
        continue
    sub_id = row[0]
    rel_id = row[2]
    obj_id = row[4]

    obj_type = row[6]
    start_time = row[7]
    end_time = row[8]

    if obj_type != 'entity' or (start_time == '' and end_time == ''):
        continue
    if start_time != '' and end_time == '':
        if start_time <= '2023-01-01':
            if start_time >= '2018-01-07':
                span_knowledge.add((sub_id, rel_id, obj_id, start_time, '2023-01-01'))
                new_e_set.add(obj_id)
                new_r_set.add(rel_id)
            else:
                span_knowledge.add((sub_id, rel_id, obj_id, '2018-01-07', '2023-01-01'))
                new_e_set.add(obj_id)
                new_r_set.add(rel_id)
    
    if start_time == '' and end_time != '':
        if end_time >= '2018-01-07':
            if end_time <= '2023-01-01':
                span_knowledge.add((sub_id, rel_id, obj_id, '2018-01-07', end_time))
                new_e_set.add(obj_id)
                new_r_set.add(rel_id)
                facts_with_nature_end.add((sub_id, rel_id, obj_id, '2018-01-07', end_time))
            else:
                span_knowledge.add((sub_id, rel_id, obj_id, '2018-01-07', '2023-01-01'))
                new_e_set.add(obj_id)
                new_r_set.add(rel_id)
    
    if start_time != '' and end_time != '':
        new_start_time, new_end_time = -1, -1
        if start_time <= '2023-01-01' and start_time >= '2018-01-07':
            new_start_time = start_time
        elif start_time < '2018-01-07':
            new_start_time = '2018-01-07'
        
        if end_time <= '2023-01-01' and end_time >= '2018-01-07':
            new_end_time = end_time
        elif end_time > '2023-01-01':
            new_end_time = '2023-01-01'
        
        if new_start_time != -1 and new_end_time != -1:
            span_knowledge.add((sub_id, rel_id, obj_id, new_start_time, new_end_time))
            new_e_set.add(obj_id)
            new_r_set.add(rel_id)
            if new_end_time != '2023-01-01':
                facts_with_nature_end.add((sub_id, rel_id, obj_id, new_start_time, new_end_time))

print(len(span_knowledge))
print(len(facts_with_nature_end))
print(len(new_e_set))
print(len(new_r_set))
print(new_e_set)
print(new_r_set)
print(list(span_knowledge)[:10])

In [ ]:
# 抓取新实体的label 和 short description
import requests
import csv
import time
import json
from tqdm import tqdm
import concurrent.futures

class WikidataLabelDescriptionFetcher:
    def __init__(self, delay=0.1, max_workers=10, batch_size=50):
        """
        初始化Wikidata标签和描述抓取器
        
        参数:
            delay: 请求间延迟（秒）
            max_workers: 最大并行工作线程数
            batch_size: 每批处理的实体数量
        """
        self.delay = delay
        self.max_workers = max_workers
        self.batch_size = batch_size
        self.headers = {
            'User-Agent': 'WikidataLabelDescriptionFetcher/1.0 (https://example.com; contact@example.com)'
        }
        
        # 结果缓存，避免重复查询
        self.result_cache = {}
    
    def fetch_entities_batch(self, entity_ids, max_retries=3):
        """
        批量获取实体标签和描述
        """
        if not entity_ids:
            return {}
            
        # 过滤已缓存的实体
        uncached_ids = [eid for eid in entity_ids if eid not in self.result_cache]
        
        if not uncached_ids:
            # 所有实体都已缓存，直接返回
            return {eid: self.result_cache[eid] for eid in entity_ids}
        
        # 将实体ID列表转换为逗号分隔的字符串
        ids_str = "|".join(uncached_ids)
        
        url = "https://www.wikidata.org/w/api.php"
        params = {
            'action': 'wbgetentities',
            'ids': ids_str,
            'format': 'json',
            'languages': 'en',
            'props': 'labels|descriptions'
        }
        
        for attempt in range(max_retries):
            try:
                response = requests.get(url, params=params, headers=self.headers, timeout=30)
                
                if response.status_code == 200:
                    data = response.json()
                    entities = data.get('entities', {})
                    
                    results = {}
                    for eid in entity_ids:
                        if eid in self.result_cache:
                            # 使用缓存的结果
                            results[eid] = self.result_cache[eid]
                        elif eid in entities:
                            # 提取标签和描述
                            entity = entities[eid]
                            
                            # 获取标签
                            labels = entity.get('labels', {})
                            label = labels.get('en', {}).get('value', '') if 'en' in labels else ''
                            
                            # 获取描述
                            descriptions = entity.get('descriptions', {})
                            description = descriptions.get('en', {}).get('value', '') if 'en' in descriptions else ''
                            
                            # 清理描述中的时间信息
                            cleaned_description = self.clean_description(description)
                            
                            result = {
                                'label': label,
                                'description': cleaned_description,
                                'original_description': description
                            }
                            
                            results[eid] = result
                            # 更新缓存
                            self.result_cache[eid] = result
                        else:
                            # 实体不存在或无法获取
                            results[eid] = {
                                'label': '',
                                'description': '',
                                'original_description': ''
                            }
                    
                    return results
                elif response.status_code == 429:
                    # 处理速率限制
                    wait_time = 10 * (attempt + 1)
                    print(f"速率限制，等待{wait_time}秒后重试")
                    time.sleep(wait_time)
                    continue
                elif response.status_code == 404:
                    print(f"部分实体不存在")
                    # 返回空结果
                    return {eid: {'label': '', 'description': '', 'original_description': ''} for eid in entity_ids}
                else:
                    print(f"HTTP错误 {response.status_code}")
                    if attempt < max_retries - 1:
                        time.sleep(5)
            except requests.exceptions.RequestException as e:
                print(f"请求异常: {e}")
                if attempt < max_retries - 1:
                    time.sleep(5)
        
        # 如果所有重试都失败，返回空结果
        return {eid: {'label': '', 'description': '', 'original_description': ''} for eid in entity_ids}
    
    def clean_description(self, description):
        """
        清理描述文本，移除时间相关的内容
        """
        if not description:
            return ""
        
        # 时间模式正则表达式
        time_patterns = [
            r'\(\s*\d{4}\s*[-–—]\s*\d{4}\s*\)',  # (1929-2003)
            r'\(\s*\d{4}\s*\)',  # (1929)
            r'from\s+\d{4}\s+to\s+\d{4}',  # from 1929 to 2003
            r'since\s+\d{4}\s+to\s+\d{4}',  # since 1929 to 2003
            r'\d{4}\s*[-–—]\s*\d{4}',  # 1929-2003 (无括号)
            r'born\s+\d{4}',  # born 1929
            r'\b\d{4}\s*-\s*\d{4}\b',  # 1929 - 2003
            r'\b\d{4}\s*–\s*\d{4}\b',  # 1929 – 2003 (不同种类的横线)
            r'\b\d{4}\s*—\s*\d{4}\b',  # 1929 — 2003
        ]
        
        cleaned = description
        
        # 应用所有时间模式
        for pattern in time_patterns:
            cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE)
        
        # 清理多余的空格和标点
        cleaned = re.sub(r'\s+', ' ', cleaned)  # 多个空格变一个
        cleaned = re.sub(r'\s*,\s*,', ',', cleaned)  # 清理多余的逗号
        cleaned = re.sub(r'^\s*[,-]\s*', '', cleaned)  # 开头是逗号或横线
        cleaned = re.sub(r'\s*[,-]\s*$', '', cleaned)  # 结尾是逗号或横线
        cleaned = cleaned.strip()
        
        return cleaned
    
    def process_entities_parallel(self, entity_ids, output_file):
        """
        使用并行处理处理实体列表
        """
        # 将实体ID分成批次
        batches = [entity_ids[i:i+self.batch_size] for i in range(0, len(entity_ids), self.batch_size)]
        
        print(f"将处理 {len(entity_ids)} 个实体，分为 {len(batches)} 批，每批 {self.batch_size} 个")
        print(f"使用 {self.max_workers} 个并行工作线程")
        
        all_results = {}
        
        # 使用线程池并行处理批次
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            # 提交所有批次任务
            future_to_batch = {executor.submit(self.fetch_entities_batch, batch): batch for batch in batches}
            
            # 收集结果
            for future in tqdm(concurrent.futures.as_completed(future_to_batch), 
                              total=len(batches), desc="处理批次"):
                batch_results = future.result()
                all_results.update(batch_results)
        
        # 保存结果到CSV文件
        self.save_results(all_results, output_file)
        
        return all_results
    
    def save_results(self, results, output_file):
        """
        保存结果到CSV文件
        """
        if not results:
            print("没有找到任何结果")
            return
        
        with open(output_file, 'a', newline='', encoding='utf-8') as f:
            fieldnames = [
                'entity_id',
                'label',
                'description',
                'original_description'
            ]
            
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            
            for entity_id, data in results.items():
                writer.writerow({
                    'entity_id': entity_id,
                    'label': data['label'],
                    'description': data['description'],
                    'original_description': data['original_description']
                })
        
        print(f"结果已保存到: {output_file}")
        print(f"共处理 {len(results)} 个实体")

def main():
    # 示例使用
    entity_ids = [
        "Q76",    # Barack Obama
        "Q90",    # Paris
        "Q114",   # Kenya
        "Q937",   # Albert Einstein
        "Q1339",  # Leonardo da Vinci
        "Q1774"   # Mozart
    ]
    
    output_file = "wikidata_labels_descriptions.csv"
    
    # 创建抓取器
    fetcher = WikidataLabelDescriptionFetcher(delay=0.1, max_workers=10, batch_size=50)
    
    # 处理实体
    results = fetcher.process_entities_parallel(entity_ids, output_file)
    
    # 打印摘要
    print("\n处理摘要:")
    print(f"处理的实体数量: {len(entity_ids)}")
    
    # 统计有标签和描述的实体数量
    with_label = sum(1 for data in results.values() if data['label'])
    with_description = sum(1 for data in results.values() if data['description'])
    
    print(f"有标签的实体数量: {with_label}")
    print(f"有描述的实体数量: {with_description}")

def load_entity_ids_from_file(file):
    """
    从文件加载实体ID列表
    """
    # 确保是有效的Wikidata ID格式
    entity_ids = []
    for entity in file:
        if entity.startswith('Q') and entity[1:].isdigit():
            entity_ids.append(entity)
        else:
            print(f"跳过无效的实体ID: {entity}")
    
    return entity_ids

if __name__ == "__main__":
    # 从文件加载实体ID
    output_file = "new_entity_labels_descriptions.csv"
    
    # 加载实体ID
    entity_ids = load_entity_ids_from_file(new_e_set)
    
    if not entity_ids:
        print("没有找到有效的实体ID")
        exit()
    
    print(f"加载了 {len(entity_ids)} 个实体ID")
    
    # 创建抓取器并处理
    fetcher = WikidataLabelDescriptionFetcher(delay=0.1, max_workers=10, batch_size=50)
    results = fetcher.process_entities_parallel(entity_ids, output_file)

In [ ]:
# 抓取新关系的label
import requests
import time
import sys

def get_property_labels(property_ids, language='en', delay=0.1, max_retries=3):
    """
    批量获取 Wikidata 属性标签
    
    参数:
        property_ids: Wikidata 属性 ID 列表 (例如 ["P31", "P21"])
        language: 需要的标签语言 (默认 'en')
        delay: 请求间延迟（秒），用于控制请求频率
        max_retries: 最大重试次数
    
    返回:
        字典: {property_id: label} 或 {property_id: None}（如果未找到）
    """
    headers = {
        'User-Agent': 'PropertyLabelFetcher/1.0 (https://example.com; contact@example.com)'
    }
    
    base_url = "https://www.wikidata.org/w/api.php"
    results = {}
    
    # 将属性ID列表分批次处理，每批50个
    batch_size = 50
    batches = [property_ids[i:i + batch_size] for i in range(0, len(property_ids), batch_size)]
    
    for batch in batches:
        params = {
            'action': 'wbgetentities',
            'ids': '|'.join(batch),
            'props': 'labels',
            'languages': language,
            'format': 'json'
        }
        
        for attempt in range(max_retries):
            try:
                response = requests.get(base_url, params=params, headers=headers, timeout=30)
                if response.status_code == 200:
                    data = response.json()
                    
                    if 'entities' in data:
                        for prop_id in batch:
                            entity = data['entities'].get(prop_id, {})
                            labels = entity.get('labels', {})
                            
                            if language in labels:
                                results[prop_id] = labels[language]['value']
                            else:
                                results[prop_id] = None
                    
                    break  # 成功则跳出重试循环
                    
                elif response.status_code == 429:
                    # 遇到速率限制，等待更长时间
                    wait_time = delay * (2 ** attempt)  # 指数退避
                    print(f"速率限制，等待 {wait_time} 秒后重试...")
                    time.sleep(wait_time)
                else:
                    print(f"HTTP 错误 {response.status_code}，尝试 {attempt + 1}/{max_retries}")
                    if attempt < max_retries - 1:
                        time.sleep(delay)
                        
            except requests.exceptions.RequestException as e:
                print(f"请求异常: {e}，尝试 {attempt + 1}/{max_retries}")
                if attempt < max_retries - 1:
                    time.sleep(delay)
        
        # 批次间延迟
        time.sleep(delay)
    
    return results

def save_results(results, output_file='new_property_labels.csv'):
    """将结果保存到 CSV 文件"""
    import csv
    
    with open(output_file, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Property_ID', 'Label'])
        
        for prop_id, label in results.items():
            writer.writerow([prop_id, label if label else 'Not Found'])
    
    print(f"结果已保存到: {output_file}")

def main():
    # 示例属性 ID 列表 - 替换为您实际的属性 ID
    property_ids = list(new_r_set)
    
    print(f"开始获取 {len(property_ids)} 个属性的标签...")
    
    # 获取属性标签
    results = get_property_labels(property_ids, language='en', delay=0.1)
    
    # 打印结果
    print("\n属性标签结果:")
    found_count = 0
    for prop_id, label in results.items():
        status = label if label else "未找到"
        print(f"{prop_id}: {status}")
        if label:
            found_count += 1
    
    print(f"\n成功获取 {found_count}/{len(property_ids)} 个属性的标签")
    
    # 保存到文件
    save_results(results)
    
    return results

if __name__ == "__main__":
    main()

In [ ]:
# 构建实体到description的字典
delete_descriptions = ['family name', 'doctoral thesis', 'scholarly article', 'scientific article', 'episode of', 'studio album', 'Journal article']

all_original_entities = set()
input_file = "entity2id.txt"
f = open(input_file, 'r', encoding='utf-8')
for line in f.readlines():
	e_text, entity_idx_original, _, _ = line.strip().split('\t')
	all_original_entities.add(e_text)

read_entity_label_description = {}
read_idx_set = set()
csv_reader = csv.reader(open("wikidata_results.csv"))
for row in csv_reader:
	if row[0] == 'original_idx_text':
		continue
	original_entity_label = eval(row[0])[1]
	read_entity_idx = row[2]
	read_entity_label = row[3]
	read_entity_des = row[4]

	if read_entity_idx != '':
		read_idx_set.add(read_entity_idx)

	for keywords in delete_descriptions:
		if keywords in read_entity_des:
			read_entity_label = ''
			read_entity_des = ''
			break
	
	if original_entity_label not in read_entity_label_description.keys():
		if read_entity_label == '' and read_entity_des == '':
			read_entity_label_description[original_entity_label] = [read_entity_idx, original_entity_label]
		elif read_entity_label != '' and read_entity_des == '':
			read_entity_label_description[original_entity_label] = [read_entity_idx, read_entity_label]
		else:
			read_entity_label_description[original_entity_label] = [read_entity_idx, read_entity_des]
	else:
		if read_entity_label == '' and read_entity_des == '':
			continue
		elif read_entity_label != '' and read_entity_des == '' and read_entity_label_description[original_entity_label] == original_entity_label:
			read_entity_label_description[original_entity_label] = [read_entity_idx, read_entity_label]
		elif read_entity_des != '' and (read_entity_label_description[original_entity_label] == original_entity_label or read_entity_label_description[original_entity_label] == read_entity_label):
			read_entity_label_description[original_entity_label] = [read_entity_idx, read_entity_des]

print(len(all_original_entities), len(read_entity_label_description.keys()), len(all_original_entities&set(read_entity_label_description.keys())))
#print(read_entity_label_description)


new_entity_label_description = {}
empty_des_set = set()
new_idx_set = set()
csv_reader = csv.reader(open("new_entity_labels_descriptions.csv"))
for row in csv_reader:
	if row[0] == 'entity_id':
		continue
	idx = row[0]
	new_idx_set.add(idx)
	read_entity_label = row[1]
	read_entity_des = row[2]
	if read_entity_label == '' and read_entity_des == '':
		empty_des_set.add(idx)
	else:
		if read_entity_label != '' and read_entity_des == '':
			new_entity_label_description[idx] = [read_entity_label, read_entity_des]
		elif read_entity_des != '' and read_entity_label == '':
			new_entity_label_description[idx] = [read_entity_label, read_entity_des]
		else:
			new_entity_label_description[idx] = [read_entity_label, read_entity_des]

print(len(new_e_set), len(new_idx_set), len(new_entity_label_description.keys())+len(empty_des_set))
#print(new_entity_label_description)

print(len(new_idx_set&read_idx_set))

final_e_2_keywords_dict = {}
projection_set = new_idx_set&read_idx_set
projection_dict = {}
for label in read_entity_label_description.keys():
	idx = read_entity_label_description[label][0]
	des = read_entity_label_description[label][1]

	if idx in projection_set:
		projection_dict[idx] = label
	
	if label not in final_e_2_keywords_dict.keys():
		final_e_2_keywords_dict[label] = des

print(len(projection_dict.keys()))
for idx in new_entity_label_description.keys():
	if idx in projection_set:
		if idx in projection_dict.keys():
			new_entity_label_description[idx] = [projection_dict[idx], new_entity_label_description[idx][1]]
		else:	
			label = new_entity_label_description[idx][0]
			des = new_entity_label_description[idx][1]
			final_e_2_keywords_dict[label] = des
	else:
		label = new_entity_label_description[idx][0]
		des = new_entity_label_description[idx][1]
		final_e_2_keywords_dict[label] = des

print(len(final_e_2_keywords_dict))
print(final_e_2_keywords_dict)

In [ ]:
# 总结知识

# 时间戳知识
id_2_rel = {}
id_2_entity = {}
id_2_time = {}

e_input_file = "entity2id.txt"
f = open(e_input_file, 'r', encoding='utf-8')
for line in f.readlines():
	e_text, e_idx, _, _ = line.strip().split('\t')
	id_2_entity[e_idx] = e_text

r_input_file = "relation2id.txt"
f = open(r_input_file, 'r', encoding='utf-8')
for line in f.readlines():
	r_text, r_idx = line.strip().split('\t')
	id_2_rel[r_idx] = r_text
	
t_input_file = "time2id.txt"
f = open(t_input_file, 'r', encoding='utf-8')
for line in f.readlines():
	t_idx, t_text = line.strip().split(',')
	id_2_time[t_idx] = t_text

print(id_2_entity)
print(id_2_rel)
print(id_2_time)


final_stamp_knowledge = set()
all_entities = set()
fact_input_files = ["train.txt", "valid.txt", "test.txt"]
for file in fact_input_files:
	f = open(file, 'r', encoding='utf-8')
	for line in f.readlines():
		s, r, o, t, _ = line.strip().split('\t')
		final_stamp_knowledge.add((id_2_entity[s], id_2_rel[r], id_2_entity[o], id_2_time[t]))
		all_entities.add(id_2_entity[s])
		all_entities.add(id_2_entity[o])
print(len(final_stamp_knowledge))
print(list(final_stamp_knowledge)[:10])



# 时间区间知识
final_span_knowlegde = set()
csv_reader = csv.reader(open("new_property_labels.csv"))
for row in csv_reader:
	if row[0] == 'Property_ID':
		continue
	id_2_rel[row[0]] = row[1]
for fact in span_knowledge:
	sub_id, rel_id, obj_id, new_start_time, new_end_time = fact
	try:
		final_span_knowlegde.add((new_entity_label_description[sub_id][0], id_2_rel[rel_id], new_entity_label_description[obj_id][0], new_start_time, new_end_time))
	except:
		pass

print(len(final_span_knowlegde))
print(list(final_span_knowlegde)[:10])

nature_end_set = set()
nature_start_set = set()
for f in final_span_knowlegde:
	if f[-1] < '2023-01-01':
		nature_end_set.add(f)
	if f[-2] > '2018-01-07':
		nature_start_set.add(f)
print(len(nature_end_set))
print(len(nature_start_set))

In [ ]:
#check
all_knowledge = []

for f in final_stamp_knowledge:
    #print(f)
    if f[0] not in final_e_2_keywords_dict.keys() or f[2] not in final_e_2_keywords_dict.keys():
        print("entity error: "+str(f))
    all_knowledge.append(f)

for f in final_span_knowlegde:
    if f[0] not in final_e_2_keywords_dict.keys() or f[2] not in final_e_2_keywords_dict.keys():
        print("entity error: "+str(f))
    if f[-2] > f[-1]:
        print("time error: "+str(f))
        continue
    all_knowledge.append(f)

print(len(all_knowledge))

In [ ]:
print(all_knowledge[-100:])

In [ ]:
import pickle
pickle.dump([final_e_2_keywords_dict, all_knowledge], open('FineWiki_e20k_f250k.pkl', 'wb'))


In [ ]:
node_2_keys, fact = pickle.load(open('FineWiki_e20k_f250k.pkl', 'rb'))
print([(key, node_2_keys[key]) for key in list(node_2_keys.keys())[:30]])
print(list(fact)[:30])

In [ ]:
import pickle
node_2_keys, fact = pickle.load(open('FinWiki_e20k_f250k.pkl', 'rb'))
print([(key, node_2_keys[key]) for key in list(node_2_keys.keys())[:30]])
print(list(fact)[:30])

In [ ]:
import matplotlib.pyplot as plt
import tqdm
from datetime import date, timedelta

from datetime import datetime, timedelta
import calendar

def get_months_between(start_date, end_date):
    """
    获取两个时间戳之间的所有月份
    
    Args:
        start_date: 开始时间戳，格式 "YYYY-MM"
        end_date: 结束时间戳，格式 "YYYY-MM"
    
    Returns:
        list: 包含所有月份的字符串列表，格式 ["YYYY-MM", ...]
    """
    # 将字符串转换为datetime对象
    try:
        start = datetime.strptime(start_date, "%Y-%m")
        end = datetime.strptime(end_date, "%Y-%m")
        
        months = []
        current = start
        
        while current <= end:
            # 将当前月份添加到列表
            months.append(current.strftime("%Y-%m"))
            
            # 计算下一个月
            # 获取当前月份的天数
            days_in_month = calendar.monthrange(current.year, current.month)[1]
            # 加上当前月份的天数，跳到下个月的第一天
            current = current + timedelta(days=days_in_month)
            # 设置为下个月的第一天
            current = current.replace(day=1)
        
        return months
    except:
        return []

# split train valid test
# 首先确定切分时间戳
all_times = set()
t_2_stamp_fact = {}
t_2_span_fact = {}
t_2_start = {}
t_2_end = {}
new_fact = set()
for f in tqdm.tqdm(fact):
    if len(f) == 4:
        s, r, o, t = f
        t = t[:-3]
        all_times.add(t)
        if t not in t_2_stamp_fact.keys():
            t_2_stamp_fact[t] = set()
        t_2_stamp_fact[t].add((s,r,o,t)) 
        new_fact.add((s,r,o,t))
    if len(f) == 5:
        s, r, o, t_s, t_e = f
        t_s = t_s[:-3]
        t_e = t_e[:-3]
        if '-00' in t_s or '-00' in t_e or ';' in t_s or ';' in t_e:
            continue
        if t_s >= t_e:
            continue
        all_times.add(t_s)
        all_times.add(t_e)
        for t in get_months_between(t_s, t_e): #range(f[-2], f[-1]+1):
            if t not in t_2_span_fact.keys():
                t_2_span_fact[t] = set()
            t_2_span_fact[t].add((s,r,o,t_s,t_e)) 
        
        if t_s not in t_2_start.keys():
            t_2_start[t_s] = set()
        t_2_start[t_s].add((s,r,o,t_s,t_e))

        if t_e not in t_2_end.keys():
            t_2_end[t_e] = set()
        t_2_end[t_e].add((s,r,o,t_s,t_e))

        new_fact.add((s,r,o,t_s,t_e))


sorted_all_times = sorted(list(all_times))
sorted_stamp_nums = []
sorted_span_nums = []
sorted_start_nums = []
sorted_end_nums = []
for t in sorted_all_times:
    if t in t_2_stamp_fact.keys():
        sorted_stamp_nums.append(len(t_2_stamp_fact[t]))
    else:
        sorted_stamp_nums.append(0)
    
    if t in t_2_span_fact.keys():
        sorted_span_nums.append(len(t_2_span_fact[t]))
    else:
        sorted_span_nums.append(0)

    if t in t_2_start.keys():
        sorted_start_nums.append(len(t_2_start[t]))
    else:
        sorted_start_nums.append(0)
    
    if t in t_2_end.keys():
        sorted_end_nums.append(len(t_2_end[t]))
    else:
        sorted_end_nums.append(0)
print(sorted_all_times)
print(len(sorted_all_times))
print(sorted_stamp_nums)
print(len(sorted_stamp_nums))
print(sorted_span_nums)
print(len(sorted_span_nums))


In [ ]:
split_ratio = 0.5
valid_preiod = 0.05 
# 切分时间戳
print('train/valid/test split point')
print([int(split_ratio*len(sorted_all_times)), sorted_all_times[int(split_ratio*len(sorted_all_times))]])
print([int((split_ratio+valid_preiod)*len(sorted_all_times)), sorted_all_times[int((split_ratio+valid_preiod)*len(sorted_all_times))]])

print('train stamp span fact num')
# 切分后的训练集时间戳知识数量
print(sum(sorted_stamp_nums[:int(split_ratio*len(sorted_all_times))]))
# 切分后的训练集时间区间知识数量
print(sum(sorted_span_nums[:int(split_ratio*len(sorted_all_times))]))

print('valid stamp span fact num')
# 切分后的验证集时间戳知识数量
print(sum(sorted_stamp_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))
# 切分后的验证集时间区间知识数量
print(sum(sorted_span_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))

print('test stamp span fact num')
# 切分后的测试集时间戳知识数量
print(sum(sorted_stamp_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):]))
# 切分后的测试集时间区间知识数量
print(sum(sorted_span_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):]))

# 检查验证集和测试集各有多少时间区间知识结束和时间区间知识开始
print('train start end fact num')
print(sum(sorted_start_nums[:int(split_ratio*len(sorted_all_times))]))
print(sum(sorted_end_nums[:int(split_ratio*len(sorted_all_times))]))
print('valid start end fact num')
print(sum(sorted_start_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))
print(sum(sorted_end_nums[int(split_ratio*len(sorted_all_times)):int((split_ratio+valid_preiod)*len(sorted_all_times))]))

print('test start end fact num')
print(sum(sorted_start_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):-1]))
print(sum(sorted_end_nums[int((split_ratio+valid_preiod)*len(sorted_all_times)):-1]))

plt.plot(sorted_stamp_nums, color='skyblue', alpha=0.5)
plt.plot(sorted_span_nums, color='red', alpha=0.5)
plt.show()

In [ ]:

# 正式切分数据集
train_set = set()
valid_set = set()
test_set = set()

train_split_time = sorted_all_times[int(split_ratio*len(sorted_all_times))-1]
valid_split_time = sorted_all_times[int((split_ratio+valid_preiod)*len(sorted_all_times))-1]
for f in new_fact:
    if len(f) == 4: # stamp knowledge
        s, r, o, t = f
        if t <= train_split_time:
            train_set.add(f)
        elif t <= valid_split_time:
            valid_set.add(f)
        else:
            test_set.add(f)
    if len(f) == 5: # span knowledge
        s, r, o, t_s, t_o = f
        if t_s <= train_split_time:
            train_set.add(f)
        elif t_s <= valid_split_time:
            valid_set.add(f)
        else:
            test_set.add(f)
        
        if t_o > train_split_time:
            valid_set.add(f)
        if t_o > valid_split_time:
            test_set.add(f)

print([len(train_set), len(valid_set), len(test_set)])
print(len(train_set|valid_set|test_set))

# check the consistency
print('train stamp span fact num')
# 切分后的训练集时间戳知识数量
print(len([f for f in train_set if len(f) == 4]))
# 切分后的训练集时间区间知识数量
print(len([f for f in train_set if len(f) == 5]))

print('valid stamp span fact num')
# 切分后的训练集时间戳知识数量
print(len([f for f in valid_set if len(f) == 4]))
# 切分后的训练集时间区间知识数量
print(len([f for f in valid_set if len(f) == 5]))

print('test stamp span fact num')
# 切分后的训练集时间戳知识数量
print(len([f for f in test_set if len(f) == 4]))
# 切分后的训练集时间区间知识数量
print(len([f for f in test_set if len(f) == 5]))

# 检查验证集和测试集各有多少时间区间知识结束和时间区间知识开始
print('train start end fact num')
start_num, end_num = 0, 0
train_start_fact = set()
train_end_fact = set()
for f in train_set:
    if len(f) == 5:
        if f[-2] <= train_split_time:
            start_num += 1
            train_start_fact.add(f)
        if f[-1] <= train_split_time:
            end_num += 1
            train_end_fact.add(f)
print([start_num, end_num])

print('valid start end fact num')
start_num, end_num = 0, 0
valid_start_fact = set()
valid_end_fact = set()
for f in valid_set:
    if len(f) == 5:
        if f[-2] <= valid_split_time and f[-2] > train_split_time:
            start_num += 1
            valid_start_fact.add(f)
        if f[-1] <= valid_split_time and f[-1] > train_split_time:
            end_num += 1
            valid_end_fact.add(f)          
print([start_num, end_num])

print('test start end fact num')
start_num, end_num = 0, 0
test_start_fact = set()
test_end_fact = set()
for f in test_set:
    if len(f) == 5:
        if f[-2] > valid_split_time and f[-2] < sorted_all_times[-1]:
            start_num += 1
            test_start_fact.add(f)
        if f[-1] < sorted_all_times[-1] and f[-1] > valid_split_time:
            end_num += 1
            test_end_fact.add(f)
print([start_num, end_num])

# check train_valid_test 泄漏
# stamp knowledge
train_stamp_facts = set([f for f in train_set if len(f) == 4])
valid_stamp_facts = set([f for f in valid_set if len(f) == 4])
test_stamp_facts = set([f for f in test_set if len(f) == 4])

print("stamp train & valid")
print(len(train_stamp_facts&valid_stamp_facts))

print("stamp valid & test")
print(len(test_stamp_facts&valid_stamp_facts))

print("stamp train & test")
print(len(train_stamp_facts&test_stamp_facts))

# span knowledge
print("start train & valid")
print(len(train_start_fact&valid_start_fact))

print("start train & test")
print(len(train_start_fact&test_start_fact))

print("start test & valid")
print(len(test_start_fact&valid_start_fact))

print("end train & valid")
print(len(train_end_fact&valid_end_fact))

print("end train & test")
print(len(train_end_fact&test_end_fact))

print("end test & valid")
print(len(test_end_fact&valid_end_fact))

In [ ]:
pickle.dump([train_split_time, valid_split_time, train_set, valid_set, test_set, node_2_keys], open('FinWiki_e20k_f250k_train_valid_test.pkl', 'wb'))